In [1]:
import os
import requests
import json
from datetime import datetime

"""
GitHub Copilot
Real life usage scenario:
- Using DAYTONA_API_KEY in a Daytona sandbox environment
- for a GitHub repo, to run AI agent tests in isolation per PR
"""


# 1. Load API key from secure secret store (GitHub Actions / local env)
DAYTONA_API_KEY = os.getenv("DAYTONA_API_KEY")
if not DAYTONA_API_KEY:
    raise SystemExit("Missing DAYTONA_API_KEY")

# 2. Repo + branch context from environment (GitHub Actions example)
GITHUB_REPOSITORY = os.getenv("GITHUB_REPOSITORY", "yourorg/yourrepo")
GITHUB_SHA = os.getenv("GITHUB_SHA", "HEAD")
PR_NUMBER = os.getenv("PR_NUMBER", "123")
BRANCH = os.getenv("GITHUB_REF", "refs/heads/main").split("/")[-1]

# Daytona API endpoint (hypothetical)
DAYTONA_API_URL = "https://api.daytona.ai/v1/sandboxes"

def create_sandbox(repo, ref, pr):
    """
    Create ephemeral Daytona sandbox for a pull request.
    """
    payload = {
        "repository": repo,
        "ref": ref,
        "pr_number": pr,
        "purpose": "ai-agent-test",
        "timeout_minutes": 45,
        "resources": {
            "cpu": "2vCPU",
            "memory": "4Gi",
            "storage_gb": 10
        },
        "commands": [
            "git checkout " + ref,
            "python -m pip install -r requirements.txt",
            "python tests/ai_agent_test.py"
        ]
    }
    headers = {
        "Authorization": f"Bearer {DAYTONA_API_KEY}",
        "Content-Type": "application/json"
    }
    r = requests.post(DAYTONA_API_URL, headers=headers, data=json.dumps(payload))
    r.raise_for_status()
    return r.json()

def get_sandbox_status(sandbox_id):
    r = requests.get(f"{DAYTONA_API_URL}/{sandbox_id}",
                     headers={"Authorization": f"Bearer {DAYTONA_API_KEY}"})
    r.raise_for_status()
    return r.json()

def delete_sandbox(sandbox_id):
    r = requests.delete(f"{DAYTONA_API_URL}/{sandbox_id}",
                        headers={"Authorization": f"Bearer {DAYTONA_API_KEY}"})
    r.raise_for_status()
    return r.status_code in (200, 204)

def main():
    print(f"[{datetime.utcnow().isoformat()}] Starting Daytona sandbox job for PR {PR_NUMBER}")
    sandbox = create_sandbox(GITHUB_REPOSITORY, BRANCH, PR_NUMBER)
    sandbox_id = sandbox["id"]
    print(f"Sandbox created: {sandbox_id}")

    # Poll until complete
    while True:
        status = get_sandbox_status(sandbox_id)
        if status["state"] in ("completed", "failed", "timed_out"):
            print("Sandbox finished:", status["state"])
            print("Logs URL:", status.get("logs_url"))
            break
        print("Sandbox still running, waiting 15s...")
        import time; time.sleep(15)

    # Evaluate results from object model test output
    result = status.get("results", {})
    if result.get("ai_agent_tests", {}).get("passed"):
        print("AI agent test PASSED")
    else:
        print("AI agent test FAILED", result.get("ai_agent_tests"))

    # Clean up sandbox
    delete_sandbox(sandbox_id)
    print("Sandbox deleted")

if __name__ == "__main__":
    main()

SystemExit: Missing DAYTONA_API_KEY

C:\Users\Vansh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
